Neden LightGBM?
Tabular Verilerde Başarı: LightGBM, özellikle çok sayıda feature içeren tabular verilerde (burada 292 önemli değişken) hızlı ve yüksek performanslı sonuçlar verir.
Hızlı Eğitim & Bellek Verimliliği: Büyük veri setleri (321.717 örnek) için eğitim süresi kısa ve bellek kullanımı optimize edilir.
Hiperparametre Esnekliği ve Early Stopping: Modelin performansını artırmak için kolayca ayarlanabilen parametreler ve early stopping gibi mekanizmalarla aşırı öğrenmenin önüne geçilebilir.



In [1]:
#-------------------------------------------------------------------------------
# random_generate_delta_e.py
#-------------------------------------------------------------------------------

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
import joblib
import os
import random

#-------------------------------------------------------------------------------
# Blok 2'den Fonksiyonlar (Jupyter Notebook'tan alındı)
#-------------------------------------------------------------------------------
def generate_random_composition(elements, min_frac=0.01, n_elements_limit=None):
    """
    Belirtilen elementler listesinden rastgele bir kompozisyon oluşturur.
    Her elementin minimum fraksiyonu min_frac ile sınırlıdır ve toplamları 1.0 olur.
    n_elements_limit: Kompozisyonda bulunacak maksimum element sayısı. None ise sınırsız.
    """
    if not elements:
        return {}

    if n_elements_limit is None or n_elements_limit > len(elements):
        n_elements_limit = len(elements)
    
    # Kompozisyonda yer alacak element sayısını rastgele belirle (en az 1)
    # n_elements_limit 0 veya negatif olamayacağı için, eğer 1 ise randint(1,1) olacak.
    if n_elements_limit <= 0: # Eğer limit anlamsızsa veya boşsa
        if elements : # Element varsa en az 1 element seç
             num_selected_elements = random.randint(1, max(1,len(elements)))
        else: # Element yoksa boş dön
            return {}
    else: # Limit anlamlıysa
        num_selected_elements = random.randint(1, n_elements_limit)


    selected_elements = random.sample(elements, num_selected_elements)
    
    if not selected_elements:
        return {}

    fractions = np.random.dirichlet(np.ones(len(selected_elements)), size=1)[0]
    
    composition = {}
    adjusted_fractions = [max(frac, min_frac) for frac in fractions]
    
    total_adjusted = sum(adjusted_fractions)
    if total_adjusted == 0: 
        if len(selected_elements) > 0:
            equal_frac = 1.0 / len(selected_elements)
            for elem in selected_elements:
                composition[elem] = max(equal_frac, min_frac)
            current_sum_final = sum(composition.values())
            if current_sum_final > 0 :
                for elem in composition:
                    composition[elem] /= current_sum_final
        return composition

    final_fractions = [f / total_adjusted for f in adjusted_fractions]

    for i, elem in enumerate(selected_elements):
        composition[elem] = final_fractions[i]
            
    current_sum_final = sum(composition.values())
    if current_sum_final > 0 and abs(current_sum_final - 1.0) > 1e-9 : 
        for elem in composition:
            composition[elem] /= current_sum_final
                
    return composition


def find_min_delta_e_random(model, elements_to_vary, all_feature_cols, scaler,
                            n_iter=10000, min_frac=0.01, fixed_features=None):
    """
    Rastgele kompozisyonlar üreterek ve modelle tahmin yaparak en düşük Delta E'yi bulur.
    fixed_features: {'feature_name': value} veya {'feature_name': "dynamic_from_comp"} 
                    şeklinde sabitlenecek/dinamik ayarlanacak ek özellikler.
    """
    min_delta_e = float('inf')
    best_composition_details = None
    
    if not elements_to_vary:
        print("Uyarı: `elements_to_vary` listesi boş. Rastgele arama yapılamıyor.")
        return min_delta_e, best_composition_details

    print(f"{n_iter} iterasyon boyunca rastgele kompozisyonlar deneniyor...")
    for i in range(n_iter):
        current_composition = generate_random_composition(elements_to_vary, min_frac=min_frac)
        if not current_composition:
            continue

        input_features = {col: 0.0 for col in all_feature_cols}
        num_elements_in_comp = 0
        for elem, frac in current_composition.items():
            if elem in input_features:
                input_features[elem] = frac
                if frac > 1e-6: # Çok küçük fraksiyonları sayma
                    num_elements_in_comp +=1
        
        if fixed_features:
            for feat, val in fixed_features.items():
                if feat in input_features:
                    if val == "dynamic_from_comp" and feat == 'comp_ntypes':
                         input_features[feat] = num_elements_in_comp
                    else:
                        input_features[feat] = val
        
        # Eğer fixed_features'ta comp_ntypes belirtilmemişse veya dynamic değilse,
        # ve comp_ntypes model özelliklerindeyse, yine de num_elements_in_comp ile ayarla.
        # Bu kısım, comp_ntypes'ın her zaman güncel olmasını sağlamaya yardımcı olur.
        if 'comp_ntypes' in input_features and \
           (not fixed_features or 'comp_ntypes' not in fixed_features or \
            (fixed_features and fixed_features.get('comp_ntypes') != "dynamic_from_comp")):
            input_features['comp_ntypes'] = num_elements_in_comp


        df_input = pd.DataFrame([input_features], columns=all_feature_cols)
        df_input_scaled = scaler.transform(df_input)
        
        try:
            predicted_delta_e = model.predict(df_input_scaled)
            current_delta_e = predicted_delta_e[0] 
        except Exception as e:
            print(f"Tahmin sırasında hata: {e}, kompozisyon: {current_composition}")
            continue 

        if current_delta_e < min_delta_e:
            min_delta_e = current_delta_e
            best_composition_details = input_features.copy() # Store a copy

        if (i + 1) % (max(1, n_iter // 10)) == 0: # n_iter 0 ise veya 10'dan küçükse max(1,0) -> 1 olur
             print(f"Iterasyon {i+1}/{n_iter}... En düşük ΔE: {min_delta_e:.4f}")
    
    if best_composition_details:
        print(f"\nEn Düşük ΔE (Rastgele Arama): {min_delta_e:.4f}")
        # Sadece element olan ve fraksiyonu > 0 olanları veya comp_ntypes gibi sabit/dinamik özellikleri göster
        display_comp = {
            k: v for k, v in best_composition_details.items() 
            if (k in elements_to_vary and v > 1e-6) or \
               (k not in elements_to_vary and v != 0.0 and (fixed_features and k in fixed_features)) or \
               (k == 'comp_ntypes' and 'comp_ntypes' in best_composition_details) # comp_ntypes'ı her zaman göster
        }
        # Eğer 'comp_ntypes' için özel bir koşul varsa veya her zaman göstermek istiyorsan, yukarıdaki koşulu düzenle
        # Örneğin, sadece elementleri ve comp_ntypes'ı göstermek için:
        # display_comp = {k: v for k, v in best_composition_details.items() if (k in elements_to_vary and v > 1e-6) or k == 'comp_ntypes'}

        print(f"Bulunan Kompozisyon (ve ilgili özellikler): {display_comp}")
    else:
        print("Rastgele arama sonucu uygun kompozisyon bulunamadı.")
            
    return min_delta_e, best_composition_details

#-------------------------------------------------------------------------------
# Ana Çalıştırma Bloğu
#-------------------------------------------------------------------------------
if __name__ == "__main__":
    MODEL_FILENAME = 'lightgbm_delta_e_model.txt'
    SCALER_FILENAME = 'lightgbm_scaler.joblib'
    FEATURES_FILENAME = 'lightgbm_model_features.joblib'

    trained_model = None
    trained_scaler = None
    model_feature_cols = None

    try:
        print(f"Model '{MODEL_FILENAME}' dosyasından yükleniyor...")
        trained_model = joblib.load(MODEL_FILENAME)
        print("Model yüklendi.")
        
        print(f"Ölçekleyici '{SCALER_FILENAME}' dosyasından yükleniyor...")
        trained_scaler = joblib.load(SCALER_FILENAME)
        print("Ölçekleyici yüklendi.")
        
        print(f"Model özellikleri '{FEATURES_FILENAME}' dosyasından yükleniyor...")
        model_feature_cols = joblib.load(FEATURES_FILENAME)
        print("Model özellikleri yüklendi.")
        
    except FileNotFoundError as e:
        print(f"HATA: Gerekli dosya bulunamadı: {e}. Lütfen modelin, ölçekleyicinin ve özellik dosyasının mevcut olduğundan emin olun.")
        exit()
    except Exception as e:
        print(f"Dosyalar yüklenirken bir hata oluştu: {e}")
        exit()

    if trained_model and trained_scaler and model_feature_cols:
        print("\n--- Rastgele Kompozisyonlarla En Düşük Delta E Arama Başlatılıyor ---")
        
        possible_elements = ['Fe', 'Ni', 'Al', 'Cr', 'Co', 'Si', 'C', 'Mo', 'W', 'V', 'Mn', 'Ti', 'Nb', 'Zr', 'Hf', 'Ta', 'Re', 'Ru', 'B', 'P', 'S']
        elements_for_search = [elem for elem in possible_elements if elem in model_feature_cols]

        if elements_for_search:
            print(f"Arama için kullanılacak elementler (modelde olanlar ve listeden seçilenler): {elements_for_search}")
            
            fixed_search_features = {}
            if 'comp_ntypes' in model_feature_cols:
                fixed_search_features['comp_ntypes'] = "dynamic_from_comp" 
            
            print(f"Rastgele arama için sabit/dinamik özellikler: {fixed_search_features}")

            # Parametreleri isteğe göre ayarla
            n_iterations = 1000  # Denenecek rastgele kompozisyon sayısı
            min_element_fraction = 0.02 # Her element için minimum fraksiyon

            min_delta_e_val, best_comp_details = find_min_delta_e_random(
                model=trained_model,
                elements_to_vary=elements_for_search,
                all_feature_cols=model_feature_cols,
                scaler=trained_scaler,
                n_iter=n_iterations,
                min_frac=min_element_fraction,
                fixed_features=fixed_search_features
            )
        else:
            print("Model özelliklerinde veya tanımlı listede arama için uygun element bulunamadı.")
    else:
        print("\nModel, ölçekleyici veya özellikler yüklenemediği için arama başlatılamıyor.")

Model 'lightgbm_delta_e_model.txt' dosyasından yükleniyor...
Model yüklendi.
Ölçekleyici 'lightgbm_scaler.joblib' dosyasından yükleniyor...
Ölçekleyici yüklendi.
Model özellikleri 'lightgbm_model_features.joblib' dosyasından yükleniyor...
Model özellikleri yüklendi.

--- Rastgele Kompozisyonlarla En Düşük Delta E Arama Başlatılıyor ---
Arama için kullanılacak elementler (modelde olanlar ve listeden seçilenler): ['Fe', 'Ni', 'Al', 'Cr', 'Co', 'Si', 'C', 'Mo', 'W', 'V', 'Mn', 'Ti', 'Nb', 'Zr', 'Hf', 'Ta', 'Re', 'Ru', 'B']
Rastgele arama için sabit/dinamik özellikler: {'comp_ntypes': 'dynamic_from_comp'}
1000 iterasyon boyunca rastgele kompozisyonlar deneniyor...
Iterasyon 100/1000... En düşük ΔE: -0.6171
Iterasyon 200/1000... En düşük ΔE: -0.7642
Iterasyon 300/1000... En düşük ΔE: -0.7642
Iterasyon 400/1000... En düşük ΔE: -0.7642
Iterasyon 500/1000... En düşük ΔE: -0.7642
Iterasyon 600/1000... En düşük ΔE: -0.7642
Iterasyon 700/1000... En düşük ΔE: -0.7642
Iterasyon 800/1000... En düşük

In [1]:

#-------------------------------------------------------------------------------
# specific_composition_delta_e.py
#-------------------------------------------------------------------------------


import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
import joblib
import os

#-------------------------------------------------------------------------------
# Blok 3'ten Fonksiyon (Jupyter Notebook'tan alındı)
#-------------------------------------------------------------------------------
def predict_delta_e_specific(composition_dict, model, all_feature_cols, scaler):
    """
    Verilen spesifik bir kompozisyon için Delta E değerini tahmin eder.
    composition_dict: {'ElementSembolü': fraksiyon, ...} ve 'comp_ntypes' gibi ek özellikleri içerebilir.
                      Bu sözlükteki elementler zaten modelin bildiği elementler olmalıdır
                      ve 'comp_ntypes' da doğru şekilde ayarlanmış olmalıdır.
    """
    print(f"\nSpesifik kompozisyon için tahmin: {composition_dict}")

    input_data_dict = {feature: 0.0 for feature in all_feature_cols}
    valid_keys_used = 0
    
    # Sadece modelin bildiği elementleri ve özellikleri al
    # comp_ntypes gibi ek özellikler de burada işlenir
    active_element_keys_in_comp = []
    for key, value in composition_dict.items():
        if key in input_data_dict:
            input_data_dict[key] = value
            valid_keys_used +=1
            if key != 'comp_ntypes' and value > 1e-6: # Eğer elementse ve fraksiyonu anlamlıysa
                 active_element_keys_in_comp.append(key)
        else:
            print(f"Uyarı: '{key}' özelliği modelin özellik listesinde bulunmuyor ve atlanacak.")

    # comp_ntypes'ın composition_dict içinde verilip verilmediğini kontrol et
    # Eğer verilmemişse ve modelde varsa, aktif element sayısına göre ayarla
    if 'comp_ntypes' in input_data_dict and 'comp_ntypes' not in composition_dict:
        input_data_dict['comp_ntypes'] = len(active_element_keys_in_comp)
        print(f"'comp_ntypes' otomatik olarak {len(active_element_keys_in_comp)} değerine ayarlandı.")
    elif 'comp_ntypes' in composition_dict and 'comp_ntypes' in input_data_dict:
        # Eğer composition_dict'te verilmişse onu kullan (zaten input_data_dict'e aktarıldı)
        pass
    elif 'comp_ntypes' in input_data_dict : # modelde var ama composition_dict'te yoksa ve yukarıdaki koşul da sağlanmadıysa
        input_data_dict['comp_ntypes'] = len(active_element_keys_in_comp)
        print(f"Uyarı: 'comp_ntypes' girdi kompozisyonunda belirtilmedi, aktif element sayısına ({len(active_element_keys_in_comp)}) göre ayarlandı.")


    if valid_keys_used == 0 or not active_element_keys_in_comp:
        print("HATA: Verilen kompozisyonda model tarafından tanınan aktif element yok. Tahmin yapılamıyor.")
        return None

    input_df = pd.DataFrame([input_data_dict], columns=all_feature_cols)
    input_df_scaled = scaler.transform(input_df)
    
    try:
        predicted_delta_e = model.predict(input_df_scaled)
        print(f"Tahmin Edilen ΔE: {predicted_delta_e[0]:.4f}")
        return predicted_delta_e[0] if predicted_delta_e is not None else None
    except Exception as e:
        print(f"Spesifik kompozisyon için tahmin sırasında hata: {e}")
        return None

#-------------------------------------------------------------------------------
# Ana Çalıştırma Bloğu
#-------------------------------------------------------------------------------
if __name__ == "__main__":
    MODEL_FILENAME = 'lightgbm_delta_e_model.txt'
    SCALER_FILENAME = 'lightgbm_scaler.joblib'
    FEATURES_FILENAME = 'lightgbm_model_features.joblib'

    trained_model = None
    trained_scaler = None
    model_feature_cols = None

    try:
        print(f"Model '{MODEL_FILENAME}' dosyasından yükleniyor...")
        trained_model = joblib.load(MODEL_FILENAME)
        print("Model yüklendi.")
        
        print(f"Ölçekleyici '{SCALER_FILENAME}' dosyasından yükleniyor...")
        trained_scaler = joblib.load(SCALER_FILENAME)
        print("Ölçekleyici yüklendi.")
        
        print(f"Model özellikleri '{FEATURES_FILENAME}' dosyasından yükleniyor...")
        model_feature_cols = joblib.load(FEATURES_FILENAME)
        print("Model özellikleri yüklendi.")
        
    except FileNotFoundError as e:
        print(f"HATA: Gerekli dosya bulunamadı: {e}. Lütfen modelin, ölçekleyicinin ve özellik dosyasının mevcut olduğundan emin olun.")
        exit()
    except Exception as e:
        print(f"Dosyalar yüklenirken bir hata oluştu: {e}")
        exit()

    if trained_model and trained_scaler and model_feature_cols:
        print("\n--- Spesifik Kompozisyonlar İçin Delta E Tahmini Başlatılıyor ---")
        
        # Örnek 1 (Notebook'taki gibi)
        specific_composition_1_elements = {'Fe': 0.6, 'Cr': 0.2, 'Ni': 0.1, 'Mo': 0.1}
        # Sadece modelin bildiği elementleri al
        comp_to_predict_1 = {k:v for k,v in specific_composition_1_elements.items() if k in model_feature_cols}
        
        # comp_ntypes'ı aktif element sayısına göre ayarla (eğer modelde varsa)
        if 'comp_ntypes' in model_feature_cols:
            element_keys_in_comp1 = [k for k in comp_to_predict_1.keys() if k != 'comp_ntypes' and comp_to_predict_1.get(k, 0) > 1e-6]
            comp_to_predict_1['comp_ntypes'] = len(element_keys_in_comp1) 

        original_element_keys_1 = [k for k in specific_composition_1_elements.keys()]
        valid_element_keys_1 = [k for k in comp_to_predict_1.keys() if k != 'comp_ntypes' and comp_to_predict_1.get(k, 0) > 1e-6]

        if valid_element_keys_1 and len(valid_element_keys_1) >= len(original_element_keys_1) * 0.5:
            predict_delta_e_specific(
                composition_dict=comp_to_predict_1,
                model=trained_model,
                all_feature_cols=model_feature_cols,
                scaler=trained_scaler
            )
        else:
            print(f"\n'{specific_composition_1_elements}' için yeterli sayıda eşleşen özellik ({len(valid_element_keys_1)}/{len(original_element_keys_1)}) bulunamadı veya hiç geçerli element yok, tahmin atlanıyor.")

        # Örnek 2 (Notebook'taki gibi)
        another_composition_elements = {'Al': 0.5, 'Ti': 0.3, 'V': 0.2}
        comp_to_predict_2 = {k:v for k,v in another_composition_elements.items() if k in model_feature_cols}

        if 'comp_ntypes' in model_feature_cols:
            element_keys_in_comp2 = [k for k in comp_to_predict_2.keys() if k != 'comp_ntypes' and comp_to_predict_2.get(k, 0) > 1e-6]
            comp_to_predict_2['comp_ntypes'] = len(element_keys_in_comp2)
        
        original_element_keys_2 = [k for k in another_composition_elements.keys()]
        valid_element_keys_2 = [k for k in comp_to_predict_2.keys() if k != 'comp_ntypes' and comp_to_predict_2.get(k, 0) > 1e-6]

        if valid_element_keys_2 and len(valid_element_keys_2) >= len(original_element_keys_2) * 0.5 :
             predict_delta_e_specific(
                composition_dict=comp_to_predict_2,
                model=trained_model,
                all_feature_cols=model_feature_cols,
                scaler=trained_scaler
            )
        else:
            print(f"\n'{another_composition_elements}' için yeterli sayıda eşleşen özellik ({len(valid_element_keys_2)}/{len(original_element_keys_2)}) bulunamadı veya hiç geçerli element yok, tahmin atlanıyor.")
        
        # Arayüzden gelen veya manuel olarak girilecek bir kompozisyon için örnek:
        # Diyelim ki arayüzden şu elementler ve oranları geldi:
        # custom_elements = {'Ni': 0.25, 'Cr': 0.25, 'Mo': 0.25, 'W': 0.25} # 'comp_ntypes' kullanıcıdan alınabilir veya hesaplanabilir
        # comp_to_predict_custom = {k:v for k,v in custom_elements.items() if k in model_feature_cols}
        # if 'comp_ntypes' in model_feature_cols:
        #     element_keys_in_custom = [k for k in comp_to_predict_custom.keys() if k != 'comp_ntypes' and comp_to_predict_custom.get(k, 0) > 1e-6]
        #     comp_to_predict_custom['comp_ntypes'] = len(element_keys_in_custom) # Ya da kullanıcıdan alınır: custom_elements.get('comp_ntypes', len(element_keys_in_custom))

        # predict_delta_e_specific(
        #     composition_dict=comp_to_predict_custom,
        #     model=trained_model,
        #     all_feature_cols=model_feature_cols,
        #     scaler=trained_scaler
        # )

    else:
        print("\nModel, ölçekleyici veya özellikler yüklenemediği için tahmin başlatılamıyor.")

Model 'lightgbm_delta_e_model.txt' dosyasından yükleniyor...
Model yüklendi.
Ölçekleyici 'lightgbm_scaler.joblib' dosyasından yükleniyor...
Ölçekleyici yüklendi.
Model özellikleri 'lightgbm_model_features.joblib' dosyasından yükleniyor...
Model özellikleri yüklendi.

--- Spesifik Kompozisyonlar İçin Delta E Tahmini Başlatılıyor ---

Spesifik kompozisyon için tahmin: {'Fe': 0.6, 'Cr': 0.2, 'Ni': 0.1, 'Mo': 0.1, 'comp_ntypes': 4}
Tahmin Edilen ΔE: -0.0826

Spesifik kompozisyon için tahmin: {'Al': 0.5, 'Ti': 0.3, 'V': 0.2, 'comp_ntypes': 3}
Tahmin Edilen ΔE: -0.0327
